# Step 1: Load + Inspect Data

In [1]:
!pip install statsforecast mlforecast neuralforecast lightgbm chronos-forecasting

In [ ]:
!pip install mlforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.8 MB/s eta 0:00:00


## Step 2 - Imports

In [2]:
import os
import numpy as np
import pandas as pd
import torch

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA

from mlforecast import MLForecast
from lightgbm import LGBMRegressor

from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS

from chronos import ChronosPipeline

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape, bias
from utilsforecast.plotting import plot_series

## Step 3: Load the hotel data

In [3]:
df = pd.read_parquet("sample_hotels-1.parquet")

df = (
    df
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

df.head()

,unique_id,ds,holiday_flag,target_day,target_month,target_year,location_type,hotel_type,y,otb_1,...,otb_51,otb_52,otb_53,otb_54,otb_55,otb_56,otb_57,otb_58,otb_59,otb_60
0,hotel_0,2022-01-01,no,Sat,Jan,2022,NonSuburban,Resorts & Destinations,0.975309,0.679012,...,0.197531,0.197531,0.197531,0.185185,0.160494,0.160494,0.160494,0.160494,0.160494,0.160494
1,hotel_0,2022-01-02,no,Sun,Jan,2022,NonSuburban,Resorts & Destinations,0.493827,0.308642,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.061728,0.061728,0.049383
2,hotel_0,2022-01-03,no,Mon,Jan,2022,NonSuburban,Resorts & Destinations,0.456790,0.358025,...,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691
3,hotel_0,2022-01-04,no,Tue,Jan,2022,NonSuburban,Resorts & Destinations,0.592593,0.419753,...,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.037037,0.037037,0.024691,0.024691
4,hotel_0,2022-01-05,no,Wed,Jan,2022,NonSuburban,Resorts & Destinations,0.530864,0.407407,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.049383,0.049383,0.024691,0.024691,0.012346


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10172 entries, 0 to 10171
Data columns (total 69 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   unique_id      10172 non-null  object        
 1   ds             10172 non-null  datetime64[us]
 2   holiday_flag   10172 non-null  object        
 3   target_day     10172 non-null  object        
 4   target_month   10172 non-null  object        
 5   target_year    10172 non-null  int32         
 6   location_type  10172 non-null  object        
 7   hotel_type     10172 non-null  object        
 8   y              10172 non-null  float64       
 9   otb_1          10172 non-null  float64       
 10  otb_2          10172 non-null  float64       
 11  otb_3          10172 non-null  float64       
 12  otb_4          10172 non-null  float64       
 13  otb_5          10172 non-null  float64       
 14  otb_6          10172 non-null  float64       
 15  otb_7          1017

## Step 4: Create output folders

In [5]:
os.makedirs("outputs", exist_ok=True)
os.makedirs("plots", exist_ok=True)

## Step 5: Create train/test split

We are forecasting the last 28 days for each hotel.

In [6]:
train = (
    df
    .groupby("unique_id", group_keys=False)
    .apply(lambda x: x.iloc[:-28])
    .reset_index(drop=True)
)

test = (
    df
    .groupby("unique_id", group_keys=False)
    .apply(lambda x: x.iloc[-28:])
    .reset_index(drop=True)
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (9640, 69)
Test shape: (532, 69)


/tmp/ipykernel_56319/2160362775.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-28])
/tmp/ipykernel_56319/2160362775.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[-28:])


## Step 6: Prepare StatsForecast data

In [7]:
from utilsforecast.preprocessing import fill_gaps

sf_train = train[["unique_id", "ds", "y"]]
sf_train = fill_gaps(sf_train, freq="D")
sf_train["y"] = sf_train.groupby("unique_id")["y"].ffill().bfill()

sf_test = test[["unique_id", "ds", "y"]].copy()
sf_test["ds"] = pd.to_datetime(sf_test["ds"])

sf_train.head()

,unique_id,ds,y
0,hotel_0,2022-01-01,0.975309
1,hotel_0,2022-01-02,0.493827
2,hotel_0,2022-01-03,0.456790
3,hotel_0,2022-01-04,0.592593
4,hotel_0,2022-01-05,0.530864


## Step 7: Define StatsForecast models

In [8]:
sf_models = [
    Naive(),
    SeasonalNaive(season_length=7),
    AutoETS(season_length=7),
    AutoARIMA(season_length=7)
]

sf = StatsForecast(
    models=sf_models,
    freq="D",
    n_jobs=-1
)

## Step 8: Run 5-fold cross-validation

This may take a while because of AutoARIMA.

In [9]:
cv_sf = sf.cross_validation(
    df=sf_train,
    h=28,
    n_windows=5,
    step_size=28
)

cv_sf.head()

,unique_id,ds,cutoff,y,Naive,SeasonalNaive,AutoETS,AutoARIMA
0,hotel_0,2023-01-14,2023-01-13,0.938272,0.790123,0.629630,0.761871,0.732111
1,hotel_0,2023-01-15,2023-01-13,0.604938,0.790123,0.358025,0.561969,0.481735
2,hotel_0,2023-01-16,2023-01-13,0.469136,0.790123,0.555556,0.568555,0.502310
3,hotel_0,2023-01-17,2023-01-13,0.567901,0.790123,0.432099,0.583771,0.491153
4,hotel_0,2023-01-18,2023-01-13,0.666667,0.790123,0.456790,0.573340,0.494599


In [10]:
cv_sf.to_csv("outputs/cv_results_statsforecast.csv", index=False)

## Step 9: Evaluate StatsForecast models

In [11]:
eval_sf = evaluate(
    df=cv_sf,
    metrics=[bias, mae, rmse, mape]
)

eval_sf.head()

,unique_id,cutoff,metric,Naive,SeasonalNaive,AutoETS,AutoARIMA
0,hotel_0,2023-01-13,bias,0.003086,-0.256173,-0.158943,-0.247447
1,hotel_105,2023-01-13,bias,0.388814,0.004268,0.137769,0.118767
2,hotel_112,2023-01-13,bias,-0.302009,-0.102902,-0.164537,-0.145869
3,hotel_126,2023-01-13,bias,0.091633,0.018782,0.038140,0.012186
4,hotel_133,2023-01-13,bias,-0.252927,-0.156909,-0.259729,-0.154314


In [12]:
eval_sf.to_csv("outputs/evaluation_statsforecast.csv", index=False)

## Step 10: Add LightGBM with MLForecast

This satisfies the ML model requirement.

In [13]:
mlf = MLForecast(
    models=[LGBMRegressor(random_state=42)],
    freq="D",
    lags=[1, 2, 3, 7, 14, 28],
    date_features=["dayofweek", "month"]
)

In [14]:
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

## Step 11: Run LightGBM cross-validation

In [15]:
ml_train = sf_train[["unique_id", "ds", "y"]].copy()
ml_train["ds"] = pd.to_datetime(ml_train["ds"])

ml_train = (
    ml_train
    .groupby(["unique_id", "ds"], as_index=False)["y"]
    .mean()
)

def complete_daily_series(group):
    hotel_id = group["unique_id"].iloc[0]
    full_dates = pd.date_range(
        start=group["ds"].min(),
        end=group["ds"].max(),
        freq="D"
    )
    group = (
        group
        .set_index("ds")
        .reindex(full_dates)
        .rename_axis("ds")
        .reset_index()
    )
    group["unique_id"] = hotel_id
    group["y"] = group["y"].ffill().bfill()
    return group

ml_train = (
    ml_train
    .groupby("unique_id", group_keys=False)
    .apply(complete_daily_series)
    .reset_index(drop=True)
)

print("Cleaned ML training shape:", ml_train.shape)

Cleaned ML training shape: (9842, 3)


/tmp/ipykernel_56319/1966410534.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(complete_daily_series)


In [16]:
# FIXED: n_windows=5 to match the 5-fold requirement
cv_lgb = mlf.cross_validation(
    df=ml_train,
    h=28,
    n_windows=5,
    step_size=28
)

cv_lgb.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000654 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 6650, number of used features: 8
[LightGBM] [Info] Start training from score 0.666922
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 7182, number of used features: 8
[LightGBM] [Info] Start training from score 0.660282
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000687 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1

,unique_id,ds,cutoff,y,LGBMRegressor
0,hotel_0,2023-01-14,2023-01-13,0.938272,0.845779
1,hotel_0,2023-01-15,2023-01-13,0.604938,0.446080
2,hotel_0,2023-01-16,2023-01-13,0.469136,0.380516
3,hotel_0,2023-01-17,2023-01-13,0.567901,0.382555
4,hotel_0,2023-01-18,2023-01-13,0.666667,0.375684


## Step 12: Evaluate LightGBM

In [17]:
eval_lgb = evaluate(
    df=cv_lgb,
    metrics=[bias, mae, rmse, mape]
)

eval_lgb.to_csv("outputs/evaluation_lightgbm.csv", index=False)
eval_lgb.head()

,unique_id,cutoff,metric,LGBMRegressor
0,hotel_0,2023-01-13,bias,-0.268596
1,hotel_105,2023-01-13,bias,0.065681
2,hotel_112,2023-01-13,bias,-0.249413
3,hotel_126,2023-01-13,bias,0.065672
4,hotel_133,2023-01-13,bias,-0.184310


## Step 13: Combine evaluation results

In [20]:
# Step 13 — NBEATS + NHITS: define models (fixed hyperparameters, faster than Auto variants)
from neuralforecast.models import NBEATS, NHITS

nf_models = [
    NBEATS(h=28, input_size=56, max_steps=100),
    NHITS(h=28, input_size=56, max_steps=100)
]

nf = NeuralForecast(
    models=nf_models,
    freq="D"
)

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


In [21]:
# Step 13 continued — 5-fold CV
cv_nf = nf.cross_validation(
    df=sf_train,
    h=28,
    n_windows=5,
    step_size=28
)

cv_nf.to_csv("outputs/cv_results_neuralforecast.csv", index=False)
cv_nf.head()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
4.8 K     Non-trainable params
2.6 M     Total params
10.231    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

,unique_id,ds,cutoff,NBEATS,NHITS,y
0,hotel_0,2023-01-14,2023-01-13,0.722356,0.760720,0.938272
1,hotel_0,2023-01-15,2023-01-13,0.425721,0.534919,0.604938
2,hotel_0,2023-01-16,2023-01-13,0.480759,0.511318,0.469136
3,hotel_0,2023-01-17,2023-01-13,0.552806,0.555264,0.567901
4,hotel_0,2023-01-18,2023-01-13,0.492388,0.561663,0.666667


## Step 14 — Evaluate NBEATS/NHITS CV


In [22]:
eval_nf = evaluate(
    df=cv_nf,
    metrics=[bias, mae, rmse, mape]
)

eval_nf.to_csv("outputs/evaluation_neuralforecast.csv", index=False)
eval_nf.head()

,unique_id,cutoff,metric,NBEATS,NHITS
0,hotel_0,2023-01-13,bias,-0.219322,-0.176207
1,hotel_105,2023-01-13,bias,0.179358,0.180033
2,hotel_112,2023-01-13,bias,-0.140183,-0.118513
3,hotel_126,2023-01-13,bias,0.084684,0.074878
4,hotel_133,2023-01-13,bias,-0.137421,-0.160129


##

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("hotel_project_outputs", "zip", "outputs")
files.download("hotel_project_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>